This section imports necessary tools and standard C++ libraries 

In [ ]:
#include <TTree.h>
#include <TFile.h>
#include <TDatabasePDG.h>
#include <TLorentzVector.h>
#include <TMath.h>
#include <TCanvas.h>
#include <TBenchmark.h>
#include <iostream>

Headers specific to ROOT and CLAS12 for reading .hipo files

In [ ]:
#include "clas12reader.h"
#include "HipoChain.h"

A shorcut that tells the compiler to automatically look inside the Standard Library (std) whenever it encounters a command it doesn't immediately recognize

In [ ]:
using namespace std;

A translation tool that takes raw data structure from CLAS12 software and converts it into a 4 vector for ROOT software to understand

p4.SetXYZM(...): This is a specific command belonging to the ROOT TLorentzVector class. It tells the vector: "I am going to give you four numbers: the X-momentum, Y-momentum, Z-momentum, and the invariant Mass. Use these to build the full 4-momentum vector

rp->par()->getPx(): This is how C++ extracts data from the CLAS12 pointer.

rp is the particle.

->par() accesses the base kinematics properties of that particle.

->getPx() fetches the specific momentum value along the X-axis in GeV/c

In [ ]:
void SetLorentzVector(TLorentzVector &p4, clas12::region_part_ptr rp)
{
    p4.SetXYZM(rp->par()->getPx(), rp->par()->getPy(), rp->par()->getPz(), p4.M());
}

void: This function does not return a new variable. It just modifies something that already exists

A correction factor is applied to account for detector efficiencies     

In [ ]:
TLorentzVector CorrectElectron(TLorentzVector &p4)

defines a function named CorrectElectron to measure momentum and energy of an electron

In [ ]:
Double_t E_cor, px_el, py_el, pz_el;
TLorentzVector el_new;

E_cor = p4.E()
      + 0.085643
      - 0.0288063 * p4.E()
      + 0.00894691 * p4.E() * p4.E()
      - 0.000725449 * p4.E() * p4.E() * p4.E();

E_cor = corrected energy
p4.E() = gets raw uncorrected energy of electron directly from 4 vector

this is a 3rd degree polynomial that represents the curve fit from plotting the difference between the true energy and measured energy

In [ ]:
px_el = E_cor * (p4.Px() / p4.Rho());
py_el = E_cor * (p4.Py() / p4.Rho());
pz_el = E_cor * (p4.Pz() / p4.Rho());

used to change the x,y,z momentum vectors
p4.Rho() calculates the magnitude of the 3-momentum

stretching the momentum vectors out to their new, correct lengths without changing the direction the electron was flying.

In [ ]:
el_new.SetXYZM(px_el, py_el, pz_el, 0.000511);
return el_new;

Building the new vector el.new

In [ ]:
struct ParticleInfo {
    int   pid;
    int   charge;
    float px;
    float py;
    float pz;
    float P_mag;
    float vx;
    float vy;
    float vz;
    float theta;
    float phi;
    float deltaTime;
    float beta;
    float betafromP;
    float path;
    int   region;
    int   status;
    float chi2pid;
};

struct = custom storage container or a blueprint for a data profile

Instead of passing around 18 different loose variables for every single electron or proton, this struct groups them all together into one neat, organized package called ParticleInfo